# Prepare Demo Data

This notebook generates the **intentionally messy** demo CSVs used by the app and `01_pipeline_demo.ipynb`. It starts from clean data and introduces:

- **Sales:** mixed date formats, missing `amount`, duplicate `order_id`, inconsistent `region` (North/NORTH/north).
- **Customer:** mixed date formats, missing `lifetime_value`, duplicate `customer_id`, inconsistent `segment` (Gold/gold/GOLD/Premium).

Run from **project root** or from `notebooks/` (first code cell sets `ROOT`).  
**Outputs:** `data/demo_sales/sales_messy.csv`, `data/demo_customer/customer_messy.csv`.  
**Note:** Saving overwrites the tracked demo CSVs under `data/`. Commit only if you mean to change the bundled demos.

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DEMO_SALES_DIR = ROOT / "data" / "demo_sales"
DEMO_CUSTOMER_DIR = ROOT / "data" / "demo_customer"

## 1. Sales demo (clean → messy)

In [ ]:
# Start from clean data (same schema as in docs)
sales_clean = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "order_date": ["2024-01-15", "2024-01-15", "2024-01-16", "2024-01-17"],
    "product_id": ["P001", "P002", "P003", "P004"],
    "amount": [120.5, 95.0, 80.0, 200.0],
    "quantity": [2, 1, 1, 2],
    "region": ["North", "North", "North", "South"],
    "customer_id": ["C001", "C002", "C003", "C004"],
})

# Introduce intentional issues
sales_messy = sales_clean.copy()

# Mixed date formats
sales_messy.loc[1, "order_date"] = "15/01/2024"
sales_messy.loc[2, "order_date"] = "01-16-2024"

# Missing amount
sales_messy.loc[1, "amount"] = pd.NA

# Inconsistent region (North / NORTH / north)
sales_messy.loc[1, "region"] = "NORTH"
sales_messy.loc[2, "region"] = "north"

# Duplicate row (same order_id)
dup = sales_messy.iloc[2:3].copy()
dup.loc[dup.index[0], "order_date"] = "2024/01/16"
dup.loc[dup.index[0], "region"] = "North"
sales_messy = pd.concat([sales_messy, dup], ignore_index=True)

sales_messy

In [ ]:
DEMO_SALES_DIR.mkdir(parents=True, exist_ok=True)
out_sales = DEMO_SALES_DIR / "sales_messy.csv"
sales_messy.to_csv(out_sales, index=False)
print(f"Saved: {out_sales}")

## 2. Customer demo (clean → messy)

In [ ]:
# Start from clean data
customer_clean = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004"],
    "signup_date": ["2023-01-05", "2023-01-05", "2023-01-07", "2023-01-10"],
    "segment": ["Gold", "Gold", "Gold", "Silver"],
    "lifetime_value": [5000, 3200, 3000, 1500],
    "tenure": [12, 8, 6, 3],
})

# Introduce intentional issues
customer_messy = customer_clean.copy()

# Mixed date formats
customer_messy.loc[1, "signup_date"] = "05/01/2023"
customer_messy.loc[2, "signup_date"] = "2023/01/07"

# Missing lifetime_value
customer_messy.loc[1, "lifetime_value"] = pd.NA

# Inconsistent segment (Gold / gold / GOLD / Premium)
customer_messy.loc[1, "segment"] = "gold"
customer_messy.loc[2, "segment"] = "GOLD"

# Duplicate row (same customer_id)
dup = customer_messy.iloc[2:3].copy()
dup.loc[dup.index[0], "signup_date"] = "07-01-2023"
dup.loc[dup.index[0], "segment"] = "Premium"
customer_messy = pd.concat([customer_messy, dup], ignore_index=True)

customer_messy

In [ ]:
DEMO_CUSTOMER_DIR.mkdir(parents=True, exist_ok=True)
out_customer = DEMO_CUSTOMER_DIR / "customer_messy.csv"
customer_messy.to_csv(out_customer, index=False)
print(f"Saved: {out_customer}")

## 3. Verify with pipeline

Optional: run detection on the generated files to confirm duplicates, invalid formats, inconsistent categories, and IQR outlier counts are detected.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.load import load_demo_sales, load_demo_customer
from src.detect import detect_issues

for name, loader in [("Sales", load_demo_sales), ("Customer", load_demo_customer)]:
    df = loader()
    issues = detect_issues(df)
    dup = issues["duplicates"]["duplicate_row_count"]
    inv = sum(issues["invalid_formats"].values())
    inc = sum(issues["inconsistent_categories"].values())
    out = issues.get("outliers", {})
    out_n = sum(v.get("count", 0) if isinstance(v, dict) else 0 for v in out.values())
    print(f"{name}: duplicates={dup}, invalid_formats={inv}, inconsistent_categories={inc}, outlier_rows_total={out_n}")